# Hyperparameter tuning
Hyperparameter tuning is the process of selecting the optimal configuration variables (hyperparameters) for a machine learning model before training begins. Because these settings are not learned from the data, proper tuning controls model complexity and directly maximizes accuracy while preventing underfitting and overfitting
**Types**
1. Grid Search
- A brute-force, exhaustive search approach. You define a specific set of values for each hyperparameter, and the algorithm evaluates the model against every possible combination of those values to find the best performer
2. Random Search
- Instead of trying every single combination, this approach randomly selects combinations of hyperparameters from your predefined ranges.
3. Bayesian Optimization
- An intelligent, guided approach that builds a probability model (surrogate model, often using Gaussian Processes or Tree-structured Parzen Estimators) to map hyperparameters to the model's performance.
4. Manual Search
- The process of a data scientist or machine learning engineer manually adjusting parameter values one by one based on intuition, domain knowledge, and the results of previous training runs

# Cross Validation
Cross-validation (CV) is a resampling technique used to evaluate how well a machine learning model generalizes to unseen data. It splits your dataset into multiple parts, using some parts to train the model and others to test it, ensuring every data point is used for both training and testing.\
This process prevents overfitting and provides a much more reliable estimate of model performance than a single train-test split.

In [2]:
# Step 1: Import required libraries
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score,confusion_matrix

# Step 2: Load the Iris dataset
iris = load_iris()
X, y = iris.data, iris.target

# Step 3: Split into training and validation/test sets
# We use 80% for training/tuning and 20% for final evaluation.
# 'stratify=y' ensures each split has an equal balance of flower types.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Step 4: Build the Machine Learning Pipeline
# Scaling is optional for Random Forests, but using a pipeline is best practice
# to safely manage data workflows and prevent information leakage.
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', RandomForestClassifier(random_state=42))
])

# Step 5: Define the Random Forest hyperparameter grid
# Note: Use 'classifier__' prefix to point to the pipeline's classifier step.
param_grid = {
    'classifier__n_estimators': [50, 100, 150],     # Number of trees in the forest
    'classifier__max_depth': [None, 3, 5, 10],       # Maximum depth of the trees
    'classifier__min_samples_split': [2, 5, 10],     # Min samples required to split a node
    'classifier__criterion': ['gini', 'entropy']     # Function to measure split quality
}

# Step 6: Set up Grid Search with 5-Fold Cross-Validation
# This calculates 3 * 4 * 3 * 2 = 72 combinations, run 5 times each (360 total fits).
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=5,                  # 5-fold cross-validation
    scoring='accuracy',    # Target optimization metric
    n_jobs=-1,             # Use all available CPU cores for parallel processing
    verbose=1              # Log progress updates to the screen
)

# Step 7: Train and tune the model
print("Starting Random Forest hyperparameter tuning on Iris dataset...")
grid_search.fit(X_train, y_train)

# Step 8: Extract and display the best configurations
print("\n--- Tuning Results ---")
print(f"Best Parameters Found: {grid_search.best_params_}")
print(f"Best Cross-Validation Accuracy: {grid_search.best_score_:.4f}")

# Step 9: Evaluate the final optimized model on unseen data
best_rf_model = grid_search.best_estimator_
y_pred = best_rf_model.predict(X_test)

print("\n--- Final Test Evaluation ---")
print(f"Test Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nDetailed Performance Report:")
print(classification_report(y_test, y_pred, target_names=iris.target_names))


Starting Random Forest hyperparameter tuning on Iris dataset...
Fitting 5 folds for each of 72 candidates, totalling 360 fits

--- Tuning Results ---
Best Parameters Found: {'classifier__criterion': 'gini', 'classifier__max_depth': None, 'classifier__min_samples_split': 10, 'classifier__n_estimators': 50}
Best Cross-Validation Accuracy: 0.9583

--- Final Test Evaluation ---
Test Accuracy: 0.9667

Detailed Performance Report:
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       1.00      0.90      0.95        10
   virginica       0.91      1.00      0.95        10

    accuracy                           0.97        30
   macro avg       0.97      0.97      0.97        30
weighted avg       0.97      0.97      0.97        30

